# Lab 08 — Tokenization và Masked Language Modeling

Tokenizer đi cùng checkpoint ALBERT, không dùng tokenizer của BERT thay thế.

In [1]:
from pathlib import Path
import os, sys
LAB_DIR = Path.cwd().resolve()
if LAB_DIR.name != 'lab08':
    LAB_DIR = LAB_DIR / 'labs' / 'lab08'
ROOT = LAB_DIR.parents[1]
os.environ['HF_HOME'] = str(ROOT / 'models' / 'huggingface')
os.environ['HF_HUB_DISABLE_XET'] = '1'
sys.path.insert(0, str(LAB_DIR))
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from pretrained_classification import *
seed_everything()
pd.set_option('display.max_colwidth', 160)
tokenizer = get_tokenizer()
print('Model:', MODEL_ID, '| Revision:', REVISION)
print(hardware_info())

D:\Project\NLP-2025\NLP-Labs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model: albert/albert-base-v2 | Revision: 8e2f239c5f8a2c0f253781ca60135db913e5c80c
{'device': 'cpu', 'cpu': 'Intel64 Family 6 Model 154 Stepping 3, GenuineIntel', 'platform': 'Windows-11-10.0.26200-SP0', 'torch': '2.14.1+cpu', 'threads': 4, 'gpu': None}


## 4–8. Pretraining, tokenizer và MLM

Pretraining học representation từ văn bản lớn; fine-tuning tiếp tục cập nhật model cho task có nhãn. ALBERT dùng MLM và Sentence Order Prediction (SOP), cùng cơ chế chia sẻ tham số giữa các layer. Tokenizer dùng SentencePiece; dấu `▁` thể hiện ranh giới từ, không phải token đứng riêng trong mọi trường hợp.

## 9. Kiểm tra MLM loss

Các bước tính tay được ghi trong `calculations.md`.

In [2]:
display(pd.DataFrame({'P(eats)': [0.70, 0.95, 0.01], 'loss = -ln(P)': -np.log([0.70, 0.95, 0.01])}))

,P(eats),loss = -ln(P)
0,0.70,0.356675
1,0.95,0.051293
2,0.01,4.605170


## 10. Calculation Exercise 2 — Tokenization

In [3]:
text = 'The cats are playing.'
encoded = tokenizer(text, return_special_tokens_mask=True)
tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'])
token_table = pd.DataFrame({'token': tokens, 'token_id': encoded['input_ids'],
                            'special_token': encoded['special_tokens_mask'],
                            'attention_mask': encoded['attention_mask']})
display(token_table)
content_tokens = [token for token, special in zip(tokens, encoded['special_tokens_mask']) if not special]
display(pd.DataFrame([{'whitespace_words': len(text.split()),
                       'tokens_including_special': len(tokens),
                       'content_subword_tokens_including_punctuation': len(content_tokens),
                       'continuation_pieces': sum(not t.startswith('▁') and any(ch.isalnum() for ch in t)
                                                  for t in content_tokens)}]))
print('Special tokens:', tokenizer.special_tokens_map)

,token,token_id,special_token,attention_mask
0,[CLS],2,1,1
1,▁the,14,0,1
2,▁cats,9471,0,1
3,▁are,50,0,1
4,▁playing,791,0,1
5,.,9,0,1
6,[SEP],3,1,1


,whitespace_words,tokens_including_special,content_subword_tokens_including_punctuation,continuation_pieces
0,4,7,5,0


Special tokens: {'bos_token': '[CLS]', 'eos_token': '[SEP]', 'unk_token': '<unk>', 'sep_token': '[SEP]', 'pad_token': '<pad>', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


**Trả lời:** `input_ids` là chỉ số token trong vocabulary. `attention_mask` bằng 1 ở token hợp lệ và 0 ở padding. Words được đếm bằng khoảng trắng; content subword tokens ở đây gồm cả dấu câu, nên hai số không nhất thiết bằng nhau. Một từ có thể có một hoặc nhiều subword. Special tokens được thêm bởi tokenizer và được đánh dấu trong bảng.

## 12. Tokenization — padding và attention mask

In [4]:
batch = tokenizer(['Good movie.', 'The cats are playing.'], padding=True,
                  truncation=True, max_length=64, return_tensors='pt')
print('tokens.keys():', list(batch.keys()))
for name, tensor in batch.items():
    print(name, tensor.tolist())
for ids, mask in zip(batch['input_ids'], batch['attention_mask']):
    print(list(zip(tokenizer.convert_ids_to_tokens(ids), mask.tolist())))
assert torch.all(batch['attention_mask'][batch['input_ids'] == tokenizer.pad_token_id] == 0)

tokens.keys(): ['input_ids', 'token_type_ids', 'attention_mask']
input_ids [[2, 254, 1308, 9, 3, 0, 0], [2, 14, 9471, 50, 791, 9, 3]]
token_type_ids [[0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]]
attention_mask [[1, 1, 1, 1, 1, 0, 0], [1, 1, 1, 1, 1, 1, 1]]
[('[CLS]', 1), ('▁good', 1), ('▁movie', 1), ('.', 1), ('[SEP]', 1), ('<pad>', 0), ('<pad>', 0)]
[('[CLS]', 1), ('▁the', 1), ('▁cats', 1), ('▁are', 1), ('▁playing', 1), ('.', 1), ('[SEP]', 1)]


**Trả lời:** `[CLS]` đứng đầu sequence; `[SEP]` đánh dấu kết thúc hoặc phân tách hai sequence; `<pad>` làm các câu trong batch có cùng chiều dài. `[MASK]` dùng trong MLM, còn `<unk>` biểu diễn nội dung tokenizer không nhận diện được. Đây là tên token thực tế của ALBERT. `token_type_ids` phân biệt hai đoạn nếu có; các câu đơn ở trên đều thuộc đoạn 0. Attention mask giúp padding không được dùng như nội dung thật khi tính attention.

## 8. Minh họa MLM bằng checkpoint pretrained

In [5]:
from transformers import AutoModelForMaskedLM
mlm = AutoModelForMaskedLM.from_pretrained(MODEL_ID, revision=REVISION).to(DEVICE).eval()
masked = tokenizer('The cat [MASK] fish.', return_tensors='pt').to(DEVICE)
with torch.no_grad():
    logits = mlm(**masked).logits
mask_position = (masked['input_ids'][0] == tokenizer.mask_token_id).nonzero().item()
probabilities = logits[0, mask_position].softmax(-1)
values, indices = probabilities.topk(5)
display(pd.DataFrame({'token': tokenizer.convert_ids_to_tokens(indices.tolist()),
                      'probability': values.cpu().tolist()}))
del mlm

Some weights of the model checkpoint at albert/albert-base-v2 were not used when initializing AlbertForMaskedLM: ['albert.pooler.bias', 'albert.pooler.weight']
- This IS expected if you are initializing AlbertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing AlbertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


,token,probability
0,▁loves,0.363007
1,▁liked,0.061265
2,▁chased,0.045205
3,▁loved,0.030263
4,▁watches,0.023477


**Nhận xét:** Bảng trên là dự đoán thật của pretrained ALBERT tại vị trí bị che. Phân phối này khác ví dụ xác suất giả định ở bài tính loss; không ép model phải dự đoán `eats` đúng bằng 0.70. MLM head dự đoán token, còn classification head trong notebook tiếp theo dự đoán nhãn cho cả review.